# V9.2b: 3-Stage Training

**The key experiment.** Can text guidance work on a strong backbone
if we train text+fusion FIRST (backbone frozen), then joint fine-tune?

| Stage | What | Backbone | LR |
|-------|------|----------|----|
| 1 | Vision pretrain on BraTS2021 | training | 5e-4 (done, V8.0) |
| **2** | **Text+Fusion+Decoder only** | **FROZEN** | **1e-4** |
| **3** | **Joint fine-tune all** | **unfrozen** | **1e-5** |

Key difference from V9.0 (10-epoch freeze that failed):
- Stage 2 runs **100 full epochs** with backbone completely frozen
- Stage 3 is a **separate run** with much lower LR (1e-5)
- Uses SeqCA (not ConcatScan) — V5.0 showed SeqCA gives better text delta


In [1]:
# ===== Setup =====
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi 2>/dev/null || echo 'No GPU'

!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache \
    mamba-ssm causal-conv1d einops \
    transformers nibabel pyyaml tqdm scipy

import os, subprocess, zipfile, time, shutil, glob, threading
REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'
DRIVE_CKPT = os.path.join(DRIVE_BASE, 'checkpoints')
os.makedirs(DRIVE_CKPT, exist_ok=True)

os.environ['DRIVE_CKPT_DIR'] = DRIVE_CKPT

git_dir = os.path.join(REPO_DIR, '.git')
if os.path.isdir(REPO_DIR) and not os.path.isdir(git_dir):
    shutil.rmtree(REPO_DIR)
if os.path.isdir(git_dir):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=True)
else:
    for attempt in range(1, 4):
        ret = subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/PlutoLei/TextMamba3D.git', REPO_DIR],
            capture_output=True, text=True)
        if ret.returncode == 0:
            break
        print(f'Clone attempt {attempt} failed')
        if os.path.isdir(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        time.sleep(5 * attempt)
    else:
        raise RuntimeError('Clone failed')
    os.chdir(REPO_DIR)

DATA_DIR = './data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    with zipfile.ZipFile(f'{DRIVE_BASE}/TextBraTS_data.zip', 'r') as zf:
        zf.extractall(os.path.dirname(DATA_DIR))
ET_CACHE = f'{DRIVE_BASE}/et_enriched.zip'
if os.path.exists(ET_CACHE):
    with zipfile.ZipFile(ET_CACHE, 'r') as zf:
        zf.extractall(DATA_DIR)
print(f'BraTS2020 data: {len([d for d in os.listdir(DATA_DIR) if d.startswith("BraTS")])} cases')

def sync_and_tag(tag):
    local_ckpt = os.path.join(REPO_DIR, 'checkpoints')
    if not os.path.exists(local_ckpt):
        return
    for f in glob.glob(os.path.join(local_ckpt, '*.pth')):
        shutil.copy2(f, os.path.join(DRIVE_CKPT, os.path.basename(f)))
    best = os.path.join(local_ckpt, 'best.pth')
    if os.path.exists(best):
        shutil.copy2(best, os.path.join(DRIVE_CKPT, f'best_{tag}.pth'))
        print(f'Tagged: best_{tag}.pth')
    last = os.path.join(local_ckpt, 'last.pth')
    if os.path.exists(last):
        shutil.copy2(last, os.path.join(DRIVE_CKPT, f'last_{tag}.pth'))
    print(f'Synced to {DRIVE_CKPT}')

# Background auto-sync
_sync_stop = threading.Event()
_sync_track = {}
def _file_is_stable(path, wait=3):
    try:
        s1 = os.path.getsize(path)
        time.sleep(wait)
        s2 = os.path.getsize(path)
        return s1 == s2 and s1 > 0
    except OSError:
        return False
def _bg_sync_loop():
    local_ckpt = os.path.join(REPO_DIR, 'checkpoints')
    while not _sync_stop.is_set():
        _sync_stop.wait(120)
        if _sync_stop.is_set():
            break
        try:
            files = glob.glob(os.path.join(local_ckpt, '*.pth'))
            synced = 0
            for f in files:
                mt = os.path.getmtime(f)
                name = os.path.basename(f)
                if name not in _sync_track or _sync_track[name] < mt:
                    if not _file_is_stable(f):
                        continue
                    shutil.copy2(f, os.path.join(DRIVE_CKPT, name))
                    _sync_track[name] = mt
                    synced += 1
            if synced > 0:
                print(f'[AutoSync] {synced} checkpoint(s) synced to Drive')
        except Exception as e:
            print(f'[AutoSync] Warning: {e}')
_sync_thread = threading.Thread(target=_bg_sync_loop, daemon=True)
_sync_thread.start()
print('Background auto-sync started (every 2 min)')
print('Setup complete')


Mounted at /content/drive
Sun Apr  5 15:31:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   28C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------

## Stage 2: Freeze Backbone, Train Text+Fusion+Decoder

100 epochs, backbone LR=0, text/fusion/decoder LR=1e-4


In [2]:
import os, glob, shutil
os.chdir(REPO_DIR)

os.environ['DRIVE_CKPT_DIR'] = DRIVE_CKPT

# Load V8.0 Stage 1 checkpoint
STAGE1_CKPT = os.path.join(DRIVE_CKPT, 'best_V8.0_stage1.pth')
assert os.path.exists(STAGE1_CKPT), f'Not found: {STAGE1_CKPT}'

# Check if Stage 2 already completed (resume support)
S2_BEST = os.path.join(DRIVE_CKPT, 'best_V9.2b_stage2.pth')
if os.path.exists(S2_BEST):
    print(f'Stage 2 already done: {S2_BEST}')
    print('Skip to Stage 3')
else:
    ckpt_dir = os.path.join(REPO_DIR, 'checkpoints')
    os.makedirs(ckpt_dir, exist_ok=True)
    for f in glob.glob(os.path.join(ckpt_dir, '*.pth')):
        os.remove(f)

    print(f'Stage 2: Freeze backbone, train text+fusion+decoder')
    print(f'From: {STAGE1_CKPT}')
    !python -u train.py \
        --config configs/autoresearch/V9.2b_3stage.yaml \
        --resume "{STAGE1_CKPT}" \
        --reset-optimizer \
        --freeze-vision-epochs 999 \
        --no-text-ratio 0.15 \
        --grad-accum 2

    sync_and_tag('V9.2b_stage2')
    print('Stage 2 complete!')


Stage 2: Freeze backbone, train text+fusion+decoder
From: /content/drive/MyDrive/TextMamba3D/checkpoints/best_V8.0_stage1.pth
2026-04-05 15:34:58.535787: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-05 15:34:58.544016: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775403298.553759    6429 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775403298.556999    6429 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:17754

In [3]:
# === Emergency Sync ===
import shutil, glob, os, subprocess
local_ckpt = os.path.join(REPO_DIR, 'checkpoints')
files = glob.glob(os.path.join(local_ckpt, '*.pth'))
if not files:
    print('No local checkpoints')
else:
    for f in sorted(files):
        name = os.path.basename(f)
        shutil.copy2(f, os.path.join(DRIVE_CKPT, name))
    subprocess.run(['sync'], check=True)
    print(f'{len(files)} files synced')


3 files synced


## Stage 3: Unfreeze All, Joint Fine-tune

50 epochs, all params unfrozen, LR=1e-5 (10x lower than Stage 2)


In [4]:
import os, glob
os.chdir(REPO_DIR)

os.environ['DRIVE_CKPT_DIR'] = DRIVE_CKPT

# Load Stage 2 best checkpoint
S2_CKPT = os.path.join(DRIVE_CKPT, 'best_V9.2b_stage2.pth')
assert os.path.exists(S2_CKPT), f'Stage 2 not done: {S2_CKPT}'

ckpt_dir = os.path.join(REPO_DIR, 'checkpoints')
for f in glob.glob(os.path.join(ckpt_dir, '*.pth')):
    os.remove(f)

print(f'Stage 3: Joint fine-tune from {S2_CKPT}')
!python -u train.py \
    --config configs/autoresearch/V9.2b_stage3_joint.yaml \
    --resume "{S2_CKPT}" \
    --reset-optimizer \
    --no-text-ratio 0.15 \
    --grad-accum 2

sync_and_tag('V9.2b')
print('Stage 3 complete!')


Stage 3: Joint fine-tune from /content/drive/MyDrive/TextMamba3D/checkpoints/best_V9.2b_stage2.pth
2026-04-05 17:35:36.472781: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-05 17:35:36.481490: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775410536.491560   58860 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775410536.494908   58860 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775410536.503539   58860 comput

## Evaluation


In [5]:
import subprocess, os
os.chdir(REPO_DIR)

ckpt = os.path.join(DRIVE_CKPT, 'best_V9.2b.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(REPO_DIR, 'checkpoints', 'best.pth')
assert os.path.exists(ckpt), f'No checkpoint: {ckpt}'
print(f'Evaluating: {ckpt}')

CONFIG = 'configs/autoresearch/V9.2b_stage3_joint.yaml'

for name, flags in [('text+TTA', ['--use-text', '--tta']), ('notext+TTA', ['--no-text', '--tta'])]:
    print()
    print('=' * 60)
    print(name)
    print('=' * 60)
    cmd = ['python', '-u', 'evaluate_full.py',
           '--config', CONFIG,
           '--checkpoint', ckpt,
           '--split', 'test', '--overlap', '0.5'] + flags
    ret = subprocess.run(cmd, capture_output=True, text=True)
    print(ret.stdout)
    if ret.returncode != 0:
        print(f'ERROR: {ret.stderr[-500:]}')

print()
print('Comparison:')
print('  V5.0 (SeqCA, scratch):     Mean=0.8479, delta=+0.55%')
print('  V8.0 (SeqCA, pretrained):  Mean=0.8753, delta=0.00%')
print('  V9.2a (ConcatScan, scratch): Mean=0.8482, delta=+0.06%')
print('  V9.2b (3-stage):           Mean=?, delta=?')


Evaluating: /content/drive/MyDrive/TextMamba3D/checkpoints/best_V9.2b.pth

text+TTA
[AutoSync] 1 checkpoint(s) synced to Drive
Loaded checkpoint: epoch=0, best_dice=0.8998249165805768
TextBraTS test: 95 samples

Evaluating 95 cases (test split)
Sliding window: patch=(128, 128, 128), overlap=0.5, text=True
TTA: 8-fold flip ensemble ENABLED

  BraTS20_Training_328: Dice=0.6811 (ET=0.3206, TC=0.7990, WT=0.9238) HD95_ET=50.70
  BraTS20_Training_028: Dice=0.7838 (ET=0.6420, TC=0.8682, WT=0.8411) HD95_ET=1.73
  BraTS20_Training_289: Dice=0.5722 (ET=0.0000, TC=0.7808, WT=0.9358) HD95_ET=nan
  BraTS20_Training_231: Dice=0.9457 (ET=0.9264, TC=0.9438, WT=0.9670) HD95_ET=1.00
  BraTS20_Training_261: Dice=0.7640 (ET=0.6203, TC=0.7227, WT=0.9488) HD95_ET=3.46
  BraTS20_Training_163: Dice=0.9204 (ET=0.8507, TC=0.9381, WT=0.9725) HD95_ET=1.00
  BraTS20_Training_345: Dice=0.8685 (ET=0.8610, TC=0.8452, WT=0.8994) HD95_ET=5.74
  BraTS20_Training_139: Dice=0.9060 (ET=0.9029, TC=0.9331, WT=0.8820) HD95_ET